# Lab 4 - Bronze Ingestion

Loads the Netflix CSV into a raw Bronze Delta table and creates controlled incremental batches for Silver testing.


In [0]:
%run ./lab4_00_config


In [0]:
from pyspark.sql import functions as F

def read_source_csv():
    last_error = None
    for path in default_source_paths:
        try:
            df = (
                spark.read
                .option("header", "true")
                .option("multiLine", "true")
                .option("quote", '"')
                .option("escape", '"')
                .csv(path)
            )
            if len(df.columns) > 0 and df.limit(1).count() > 0:
                print(f"Read source data from {path}")
                return df
        except Exception as exc:
            last_error = exc
            print(f"Could not read {path}: {exc}")
    raise RuntimeError(f"No source CSV path could be read. Last error: {last_error}")

source_df = read_source_csv()
display(source_df.limit(10))


In [0]:
raw_columns = [
    "show_id", "type", "title", "director", "cast", "country",
    "date_added", "release_year", "rating", "duration", "listed_in", "description"
]

source_with_metadata = source_df.select(
    *[F.col(c).cast("string").alias(c) for c in raw_columns],
    F.col("_metadata.file_path").alias("_source_file")
)

bronze_batch_1 = (
    source_with_metadata
    .withColumn("_batch_id", F.lit("batch_001_initial"))
    .withColumn("_ingestion_sequence", F.lit(1).cast("long"))
    .withColumn("_source_system", F.lit("dataset_example"))
    .withColumn("_ingested_at", F.current_timestamp())
    .withColumn("_raw_hash", F.sha2(F.concat_ws("||", *[F.coalesce(F.col(c), F.lit("")) for c in raw_columns]), 256))
)

(
    bronze_batch_1.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(bronze_table)
)

print("Initial Bronze rows:", spark.table(bronze_table).count())


In [0]:
# Batch 2 simulates duplicates and one intentional business update.
changed_keys = ["s1"]
batch_2_keys = [f"s{i}" for i in range(1, 101)]

batch_2_base = (
    bronze_batch_1
    .filter(F.col("show_id").isin(batch_2_keys))
    .withColumn("_batch_id", F.lit("batch_002_updates_duplicates"))
    .withColumn("_ingestion_sequence", F.lit(2).cast("long"))
    .withColumn("_ingested_at", F.current_timestamp())
)

changed_row = (
    bronze_batch_1
    .filter(F.col("show_id") == "s1")
    .withColumn("rating", F.lit("PG"))
    .withColumn("description", F.lit("Updated description for Lab 4 SCD testing."))
    .withColumn("_batch_id", F.lit("batch_002_updates_duplicates"))
    .withColumn("_ingestion_sequence", F.lit(3).cast("long"))
    .withColumn("_ingested_at", F.current_timestamp())
)

duplicate_rows = (
    bronze_batch_1
    .filter(F.col("show_id").isin("s2", "s3"))
    .withColumn("_batch_id", F.lit("batch_002_updates_duplicates"))
    .withColumn("_ingestion_sequence", F.lit(2).cast("long"))
    .withColumn("_ingested_at", F.current_timestamp())
)

bronze_batch_2 = (
    batch_2_base
    .unionByName(changed_row)
    .unionByName(duplicate_rows)
    .withColumn("_raw_hash", F.sha2(F.concat_ws("||", *[F.coalesce(F.col(c), F.lit("")) for c in raw_columns]), 256))
)

(
    bronze_batch_2.write
    .format("delta")
    .mode("append")
    .saveAsTable(bronze_table)
)

print("Bronze rows after batch 2:", spark.table(bronze_table).count())


In [0]:
# Batch 3 introduces a controlled new source column for schema evolution.
# It excludes keys changed in batch 2, so schema evolution cannot roll back the SCD update.
batch_3_keys = [f"s{i}" for i in range(101, 151)]

bronze_batch_3 = (
    bronze_batch_1
    .filter(F.col("show_id").isin(batch_3_keys))
    .withColumn("content_language", F.when(F.col("country") == "United States", F.lit("English")).otherwise(F.lit("Unknown")))
    .withColumn("_batch_id", F.lit("batch_003_new_column"))
    .withColumn("_ingestion_sequence", F.lit(4).cast("long"))
    .withColumn("_ingested_at", F.current_timestamp())
)

(
    bronze_batch_3.write
    .format("delta")
    .mode("append")
    .option("mergeSchema", "true")
    .saveAsTable(bronze_table)
)

print("Bronze schema after controlled evolution:")
spark.table(bronze_table).printSchema()


In [0]:
# Batch 4 is a controlled bad record for the data-quality gate.
# It proves that invalid rows are rejected and written to quarantine.
bronze_batch_4 = (
    bronze_batch_1
    .filter(F.col("show_id") == "s151")
    .withColumn("show_id", F.lit(None).cast("string"))
    .withColumn("type", F.lit("Documentary"))
    .withColumn("title", F.lit(""))
    .withColumn("release_year", F.lit("not_a_year"))
    .withColumn("content_language", F.lit(None).cast("string"))
    .withColumn("_batch_id", F.lit("batch_004_bad_quality_row"))
    .withColumn("_ingestion_sequence", F.lit(5).cast("long"))
    .withColumn("_ingested_at", F.current_timestamp())
    .withColumn("_raw_hash", F.sha2(F.concat_ws("||", *[F.coalesce(F.col(c), F.lit("")) for c in raw_columns]), 256))
)

(
    bronze_batch_4.write
    .format("delta")
    .mode("append")
    .option("mergeSchema", "true")
    .saveAsTable(bronze_table)
)

print("Bronze rows after batch 4:", spark.table(bronze_table).count())
